# 🚀 Day 7: Quantum Fourier Transform

## 14-Day Quantum DevRel Bootcamp

**Goal:** Build the QFT circuit from scratch, implement Quantum Phase Estimation, and understand the connection to Shor's algorithm.

**Today's Deliverables:**
1. ✅ Classical DFT → QFT correspondence
2. ✅ QFT circuit construction (Hadamard + controlled rotations)
3. ✅ QFT on periodic states — periodicity → frequency peaks
4. ✅ Quantum Phase Estimation (QPE) implementation
5. ✅ QPE on T, S, Z, and Rz gates
6. ✅ Connection: QPE → Shor's factoring algorithm

**Key insight:** The QFT converts periodicity in the computational basis to peaks in the Fourier basis, using only $O(n^2)$ gates. Combined with QPE, this is the engine behind Shor's algorithm and quantum chemistry.

---

## Section 1: The QFT Matrix — Classical to Quantum

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

print("✅ All imports loaded!")
print()

# ============================================================
# Section 1: DFT and QFT Matrices
# ============================================================

def classical_dft_matrix(N):
    """Build the N×N DFT matrix."""
    omega = np.exp(2j * np.pi / N)
    j, k = np.meshgrid(range(N), range(N))
    return omega ** (j * k) / np.sqrt(N)

def qft_matrix(n_qubits):
    """QFT matrix = DFT matrix with N = 2ⁿ."""
    return classical_dft_matrix(2 ** n_qubits)

print("📐 QFT Matrix Properties")
print("=" * 55)
print()

for n in [1, 2, 3]:
    Q = qft_matrix(n)
    N = 2 ** n
    is_unitary = np.allclose(Q @ Q.conj().T, np.eye(N))
    is_symmetric = np.allclose(Q, Q.T)
    
    # QFT|0⟩ = uniform
    e0 = np.zeros(N, dtype=complex)
    e0[0] = 1
    result = Q @ e0
    uniform_check = np.allclose(np.abs(result), 1/np.sqrt(N))
    
    print(f"  n={n} (N={N}): unitary={is_unitary}, symmetric={is_symmetric}, QFT|0⟩=uniform: {uniform_check}")

print()
# Show the 2-qubit QFT matrix
Q2 = qft_matrix(2)
print("QFT matrix (2 qubits):")
for row in Q2:
    print("  [", ", ".join(f"{x.real:+.2f}{x.imag:+.2f}i" for x in row), "]")

print()
print("💡 Key: Each column j is the state QFT|j⟩.")
print("   All entries have magnitude 1/√N — equal probabilities,")
print("   different phases. The phases encode frequency information.")

---
## Section 2: QFT Circuit Construction

The QFT circuit uses:
- **Hadamard gates:** Create the base frequency components
- **Controlled phase rotations $R_k$:** Add fine phase structure
- **SWAP gates:** Reverse qubit order (big-endian convention)

$$R_k = \begin{pmatrix} 1 & 0 \\ 0 & e^{2\pi i / 2^k} \end{pmatrix}$$

Total gates: $n$ Hadamards + $n(n-1)/2$ controlled rotations = $O(n^2)$

In [ ]:
# ============================================================
# Section 2: QFT Circuit
# ============================================================

def qft_circuit(n_qubits, swap=True):
    """Build the QFT circuit."""
    qc = QuantumCircuit(n_qubits)
    for j in range(n_qubits):
        qc.h(j)
        for k in range(1, n_qubits - j):
            angle = 2 * np.pi / (2 ** (k + 1))
            qc.cp(angle, j + k, j)
    if swap:
        for i in range(n_qubits // 2):
            qc.swap(i, n_qubits - 1 - i)
    return qc

def inverse_qft_circuit(n_qubits, swap=True):
    """QFT† — the inverse."""
    return qft_circuit(n_qubits, swap).inverse()

print("🔧 QFT Circuit Construction")
print("=" * 55)
print()

# Show circuits for different sizes
for n in [2, 3, 4]:
    qc = qft_circuit(n)
    gate_count = qc.size()
    depth = qc.depth()
    print(f"{n}-qubit QFT: {gate_count} gates, depth {depth}")
    if n <= 3:
        print(qc.draw())
    print()

# Verify circuit matches matrix
for n in [2, 3, 4]:
    qc = qft_circuit(n)
    op_circuit = Operator(qc)
    op_matrix = Operator(qft_matrix(n))
    match = op_circuit.equiv(op_matrix)
    print(f"  {n}-qubit: circuit matches matrix: {match}")

print()
# Gate count scaling
print("Gate count scaling:")
print(f"  {'n':>3} {'N':>8} {'QFT gates':>10} {'Classical FFT':>15}")
for n in [2, 4, 8, 10, 16, 20]:
    N = 2**n
    qft_gates = n * (n + 1) // 2 + n // 2  # H + CP + SWAPs
    fft_ops = N * n  # O(N log N)
    print(f"  {n:>3} {N:>8,} {qft_gates:>10} {fft_ops:>15,}")

---
## Section 3: QFT on Basis States and Periodic States

The QFT transforms:
- **Basis state $|j\rangle$** → equal-probability superposition with phase pattern
- **Periodic state (period $r$)** → peaks at multiples of $N/r$

The periodicity-to-peaks conversion is the mechanism behind Shor's algorithm.

In [ ]:
# ============================================================
# Section 3: QFT on States
# ============================================================

print("🔊 QFT on Computational Basis States (3 qubits)")
print("=" * 55)
print()

n = 3
N = 2 ** n
Q = qft_matrix(n)

for j in range(N):
    ej = np.zeros(N, dtype=complex)
    ej[j] = 1
    result = Q @ ej
    probs = np.abs(result) ** 2
    phases = np.angle(result) / np.pi  # in units of π
    print(f"  QFT|{j:03b}⟩: probs=[{', '.join(f'{p:.3f}' for p in probs)}]")
    print(f"           phases/π=[{', '.join(f'{ph:+.2f}' for ph in phases)}]")

print()
print("💡 All basis states map to UNIFORM probability distributions.")
print("   The information is entirely in the PHASES.")
print("   This is why direct measurement after QFT is useless —")
print("   you need structure (like QPE) to extract the phase info.")

In [ ]:
# ============================================================
# QFT on Periodic States — The Shor Mechanism
# ============================================================

print("📊 QFT on Periodic States: Periodicity → Frequency Peaks")
print("=" * 55)
print()

n = 3
N = 2 ** n
Q = qft_matrix(n)

for period in [1, 2, 4, 8]:
    # Build periodic state
    state = np.zeros(N, dtype=complex)
    positions = list(range(0, N, period))
    amplitude = 1.0 / np.sqrt(len(positions))
    for pos in positions:
        state[pos] = amplitude
    
    # Apply QFT
    result = Q @ state
    probs = np.abs(result) ** 2
    peaks = [i for i, p in enumerate(probs) if p > 0.05]
    
    input_str = '+'.join(f'|{p:03b}⟩' for p in positions)
    peak_str = ', '.join(f'|{p:03b}⟩' for p in peaks)
    
    print(f"  Period r={period}:")
    print(f"    Input:  {input_str}")
    print(f"    Output peaks: {peak_str}")
    print(f"    Expected peaks at multiples of N/r = {N}/{period} = {N//period}")
    print(f"    Probs: [{', '.join(f'{p:.3f}' for p in probs)}]")
    print()

In [ ]:
# ============================================================
# Plotly: Periodicity → Frequency Peaks Visualization
# ============================================================

n_viz = 4  # 4 qubits = 16 states for clearer visualization
N_viz = 2 ** n_viz
Q_viz = qft_matrix(n_viz)
labels = [f'|{i:0{n_viz}b}⟩' for i in range(N_viz)]

periods = [1, 2, 4, 8]
fig = make_subplots(
    rows=len(periods), cols=2,
    subplot_titles=[f for p in periods for f in [f'Input (period={p})', f'QFT Output (period={p})']],
    vertical_spacing=0.08
)

for idx, period in enumerate(periods):
    row = idx + 1
    
    # Build periodic state
    state = np.zeros(N_viz, dtype=complex)
    positions = list(range(0, N_viz, period))
    amp = 1.0 / np.sqrt(len(positions))
    for pos in positions:
        state[pos] = amp
    
    input_probs = np.abs(state) ** 2
    output_probs = np.abs(Q_viz @ state) ** 2
    
    fig.add_trace(
        go.Bar(x=labels, y=input_probs, marker_color='#636EFA', showlegend=False),
        row=row, col=1
    )
    fig.add_trace(
        go.Bar(x=labels, y=output_probs, marker_color='#EF553B', showlegend=False),
        row=row, col=2
    )

fig.update_layout(
    height=300 * len(periods), width=1000,
    title_text='QFT: Periodicity in Input → Peaks in Output',
    showlegend=False
)
fig.show()

---
## Section 4: Quantum Phase Estimation (QPE)

QPE estimates the eigenvalue phase $\phi$ of a unitary $U$:

$$U|\psi\rangle = e^{2\pi i \phi}|\psi\rangle \quad \Rightarrow \quad \text{QPE outputs } \tilde{\phi} \approx \phi$$

### QPE Circuit
1. $n$ counting qubits in $|0\rangle$, eigenstate $|\psi\rangle$ in target register
2. Hadamard on all counting qubits
3. Controlled-$U^{2^j}$ from counting qubit $j$ to target
4. Inverse QFT on counting register
5. Measure counting register → $n$-bit binary estimate of $\phi$

In [ ]:
# ============================================================
# Section 4: Quantum Phase Estimation
# ============================================================

def qpe_circuit(unitary_matrix, n_counting, eigenstate_prep=None):
    """Build QPE circuit."""
    n_target = int(np.log2(unitary_matrix.shape[0]))
    n_total = n_counting + n_target
    qc = QuantumCircuit(n_total, n_counting)
    
    # H on counting qubits
    qc.h(range(n_counting))
    
    # Prepare eigenstate
    if eigenstate_prep is not None:
        qc.compose(eigenstate_prep, qubits=range(n_counting, n_total), inplace=True)
    
    # Controlled-U^(2^j)
    u_op = Operator(unitary_matrix)
    for j in range(n_counting):
        u_power = u_op.power(2 ** j)
        cu = u_power.to_instruction().control(1)
        qc.append(cu, [j] + list(range(n_counting, n_total)))
    
    # Inverse QFT on counting register
    iqft = inverse_qft_circuit(n_counting)
    qc.compose(iqft, qubits=range(n_counting), inplace=True)
    
    # Measure
    qc.measure(range(n_counting), range(n_counting))
    return qc

def estimate_phase(unitary_matrix, eigenstate_vec, n_counting=4, shots=1024):
    """Run QPE and return estimated phase."""
    n_target = int(np.log2(unitary_matrix.shape[0]))
    prep = QuantumCircuit(n_target)
    prep.initialize(Statevector(eigenstate_vec))
    
    qc = qpe_circuit(unitary_matrix, n_counting, prep)
    sim = AerSimulator()
    counts = sim.run(qc, shots=shots).result().get_counts()
    
    best = max(counts, key=counts.get)
    measured_int = int(best, 2)
    return measured_int / (2 ** n_counting)

print("⚡ Quantum Phase Estimation")
print("=" * 55)
print()

eigenstate_1 = np.array([0, 1], dtype=complex)  # |1⟩

# QPE on standard gates
gates = {
    'T': (np.array([[1, 0], [0, np.exp(1j*np.pi/4)]]), 1/8),
    'S': (np.array([[1, 0], [0, 1j]]), 1/4),
    'Z': (np.array([[1, 0], [0, -1]]), 1/2),
}

print(f"{'Gate':>4} {'Exact φ':>10} {'QPE φ':>10} {'Error':>10} {'Qubits':>8}")
print("-" * 50)

for name, (matrix, exact_phi) in gates.items():
    for n_c in [3, 5]:
        est = estimate_phase(matrix, eigenstate_1, n_counting=n_c, shots=2048)
        err = abs(est - exact_phi)
        print(f"{name:>4} {exact_phi:>10.4f} {est:>10.4f} {err:>10.4f} {n_c:>8}")

print()
print("💡 Exact binary fractions (1/8, 1/4, 1/2) give PERFECT results.")
print("   More counting qubits improve precision for other phases.")

In [ ]:
# ============================================================
# QPE Precision Analysis
# ============================================================

print("📊 QPE Precision vs Number of Counting Qubits")
print("=" * 55)
print()

# Test with Rz(π/3): exact phase = π/3 / (4π) = 1/12 ≈ 0.08333...
# Not an exact binary fraction — requires many qubits for precision
theta = np.pi / 3
Rz = np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])
exact_phase = (theta / (4 * np.pi)) % 1.0

print(f"Testing Rz(π/3) on |1⟩: exact phase = {exact_phase:.6f}")
print(f"Binary: 0.{exact_phase:.20f}... (non-terminating)")
print()

n_counting_values = [2, 3, 4, 5, 6, 7, 8]
estimates = []
errors = []

for n_c in n_counting_values:
    est = estimate_phase(Rz, eigenstate_1, n_counting=n_c, shots=4096)
    err = min(abs(est - exact_phase), abs(est - exact_phase - 1), abs(est - exact_phase + 1))
    estimates.append(est)
    errors.append(err)
    precision = 1 / (2**n_c)
    print(f"  n={n_c}: estimate={est:.6f}, error={err:.6f}, precision=1/{2**n_c}={precision:.6f}")

print()
print("💡 Error decreases roughly as 1/2ⁿ — each qubit doubles precision.")

In [ ]:
# ============================================================
# Plotly: QPE Precision Visualization
# ============================================================

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['QPE Estimates vs Exact Phase', 'Error vs Counting Qubits']
)

# Left: estimates converging to exact value
fig2.add_trace(
    go.Scatter(x=n_counting_values, y=estimates,
               mode='lines+markers', name='QPE estimate',
               marker=dict(size=10, color='#EF553B'),
               line=dict(width=2)),
    row=1, col=1
)
fig2.add_hline(y=exact_phase, line_dash='dash', line_color='green',
               annotation_text=f'exact={exact_phase:.4f}', row=1, col=1)

# Right: error on log scale
fig2.add_trace(
    go.Scatter(x=n_counting_values, y=errors,
               mode='lines+markers', name='Error',
               marker=dict(size=10, color='#636EFA'),
               line=dict(width=2)),
    row=1, col=2
)
# Theoretical 1/2^n bound
theoretical = [1/(2**n) for n in n_counting_values]
fig2.add_trace(
    go.Scatter(x=n_counting_values, y=theoretical,
               mode='lines', name='1/2ⁿ bound',
               line=dict(dash='dash', color='gray')),
    row=1, col=2
)

fig2.update_yaxes(type='log', title_text='Error', row=1, col=2)
fig2.update_xaxes(title_text='Counting qubits', row=1, col=1)
fig2.update_xaxes(title_text='Counting qubits', row=1, col=2)
fig2.update_yaxes(title_text='Estimated phase', row=1, col=1)

fig2.update_layout(
    height=450, width=1000,
    title_text='QPE Precision: More Counting Qubits → Better Estimates',
    showlegend=True
)
fig2.show()

---
## Section 5: The Road to Shor's Algorithm

The chain of reductions:

$$\text{Factoring} \xrightarrow{\text{reduce}} \text{Order Finding} \xrightarrow{\text{QPE}} \text{Phase Estimation} \xrightarrow{\text{uses}} \text{QFT}$$

1. **Factoring → Order finding:** Find smallest $r$ with $a^r \equiv 1 \pmod{N}$
2. **Order finding → QPE:** Estimate eigenvalues of $U_a|x\rangle = |ax \bmod N\rangle$
3. **QPE → QFT:** Inverse QFT decodes the phase register
4. **Classical post-processing:** Continued fractions extract $r$ from measured $s/r$

In [ ]:
# ============================================================
# Section 5: Shor's Algorithm Connection
# ============================================================

print("🔐 Shor's Algorithm: QPE → Factoring")
print("=" * 55)
print()

# Demonstrate order finding for N=15
N_factor = 15
a = 7

print(f"Factoring N = {N_factor} using a = {a}")
print()
print("Step 1: Find order r (smallest r with a^r ≡ 1 mod N)")
for r in range(1, N_factor):
    result = pow(a, r, N_factor)
    print(f"  {a}^{r} mod {N_factor} = {result}", end="")
    if result == 1:
        print(f"  ← order r = {r}!")
        break
    print()

print()
print(f"Step 2: Compute gcd(a^(r/2) ± 1, N)")
half_power = pow(a, r // 2, N_factor)
from math import gcd
factor1 = gcd(half_power + 1, N_factor)
factor2 = gcd(half_power - 1, N_factor)
print(f"  a^(r/2) = {a}^{r//2} mod {N_factor} = {half_power}")
print(f"  gcd({half_power} + 1, {N_factor}) = gcd({half_power + 1}, {N_factor}) = {factor1}")
print(f"  gcd({half_power} - 1, {N_factor}) = gcd({half_power - 1}, {N_factor}) = {factor2}")
print(f"  {N_factor} = {factor1} × {factor2} ✓")

print()
print("Step 3: QPE estimates s/r for the modular exponentiation unitary")
print(f"  Eigenvalues of U_a: e^(2πis/{r}) for s = 0, 1, ..., {r-1}")
for s in range(r):
    phase = s / r
    print(f"  s={s}: φ = {s}/{r} = {phase:.4f}")

print()
print("The quantum speedup: O(n³) quantum vs sub-exponential classical.")
print("For RSA-2048 (n ≈ 2048): quantum needs ~4096 qubits + error correction.")
print("Current largest: ~1200 qubits (IBM). Still a way to go!")

---
## Key Takeaways

| Concept | Summary |
| :--- | :--- |
| **QFT matrix** | DFT with $N = 2^n$: $(\text{QFT})_{kj} = \omega^{jk}/\sqrt{N}$ |
| **QFT circuit** | $O(n^2)$ gates (exponentially fewer than classical FFT) |
| **Periodicity detection** | Period $r$ in input → peaks at multiples of $N/r$ in output |
| **QPE** | Estimates eigenvalue phase $\phi$ with precision $2^{-n}$ |
| **Shor's connection** | Factoring → order finding → QPE → QFT |
| **Practical limitation** | QFT is efficient; the oracle (modular expo) is the bottleneck |

**Week 1 complete!** Next: Day 8 — Variational Quantum Algorithms (VQE, QAOA), the dominant paradigm for near-term quantum applications.